# 02 · Baselines and Gate G1
1. Score the three simple baselines properly on validation and pick the **best baseline per horizon**.
2. Check both leakage tests passed (`python -m pytest tests -v -s`, run in the terminal first).
3. Fit one quick LightGBM and see whether it beats the best baseline.
4. Time every model on a small sample and project the total compute; Gate G1 passes only if all four checks pass.

In [ ]:
import sys, os, json, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
from src import evaluate as ev, metrics as M, models as mo
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10, "axes.titleweight": "bold"})
def savefig(fig, name):
    fig.tight_layout(); fig.savefig(ev.FIGS / f"{name}.png", dpi=150); plt.show()
print("results folder:", ev.RESULTS, "| smoke mode:", mo.SMOKE, "| threads:", mo.N_JOBS)

results folder: C:\dev\traffic-prediction\results | smoke mode: False | threads: 11


## 1. Baselines on validation (2018-07 to 2018-12)

In [ ]:
best_b, rows = {}, []
for h in ev.HORIZONS:
    va = ev.load_table(h, ["val"])
    t = ev.baseline_table(va)
    best_b[h] = t["MAE"].idxmin()
    for name, r in t.iterrows():
        rows.append({"horizon_min": h, "baseline": name, "MAE": r["MAE"], "RMSE": r["RMSE"], "WAPE_%": r["WAPE"] * 100, "rows": int(r["n"])})
bt = pd.DataFrame(rows)
display(bt.round(2))
bt.to_csv(ev.RESULTS / "baselines_val.csv", index=False)
(ev.RESULTS / "best_baseline.json").write_text(json.dumps({str(h): b for h, b in best_b.items()}, indent=2))
print("Best baseline per horizon (chosen on validation):", best_b)

,horizon_min,baseline,MAE,RMSE,WAPE_%,rows
0,15,Persistence,66.19,99.65,7.44,350587
1,15,Last week,101.90,179.52,11.46,350587
2,15,4-week average,89.22,152.05,10.03,350587
3,60,Persistence,143.18,200.21,16.10,350572
4,60,Last week,101.95,179.62,11.46,350572
5,60,4-week average,89.18,152.03,10.03,350572


Best baseline per horizon (chosen on validation): {15: 'Persistence', 60: '4-week average'}


### 1b. Same table on clean rows only
Rows whose target day is a repeated-template day (src/quality.py) are removed for every baseline alike.
The best baseline is still chosen on all validation rows.

In [ ]:
rows = []
for h in ev.HORIZONS:
    va = ev.load_table(h, ["val"])
    mk = ev.clean_mask(va, last="2018-12-31 23:45")
    t = ev.baseline_table(va, mk)
    for name, r in t.iterrows():
        rows.append({"horizon_min": h, "baseline": name, "MAE_clean": r["MAE"], "WAPE_clean_%": r["WAPE"] * 100, "rows_kept_%": mk.mean() * 100})
btc = pd.DataFrame(rows)
display(bt.merge(btc, on=["horizon_min", "baseline"]).round(2))
btc.to_csv(ev.RESULTS / "baselines_val_clean.csv", index=False)

,horizon_min,baseline,MAE,RMSE,WAPE_%,rows,MAE_clean,WAPE_clean_%,rows_kept_%
0,15,Persistence,66.19,99.65,7.44,350587,70.33,7.78,88.15
1,15,Last week,101.90,179.52,11.46,350587,113.56,12.57,88.15
2,15,4-week average,89.22,152.05,10.03,350587,96.66,10.70,88.15
3,60,Persistence,143.18,200.21,16.10,350572,146.38,16.20,88.15
4,60,Last week,101.95,179.62,11.46,350572,113.61,12.57,88.15
5,60,4-week average,89.18,152.03,10.03,350572,96.60,10.69,88.15


## 2. Leakage tests (Gate G1, part 1)
Run in the terminal: `python -m pytest tests -v -s` (takes a few minutes). This cell reads the result.

In [ ]:
sf = ev.RESULTS / "test_status.json"
tests_ok = False
if sf.exists():
    st = json.loads(sf.read_text()); print(st)
    tests_ok = bool(st["all_passed"]) and st["tests_collected"] >= 5
else:
    print("No test status yet. Run in the terminal:  python -m pytest tests -v -s")
print("Leakage tests passed:", tests_ok)

{'all_passed': True, 'tests_collected': 6, 'tests_failed': 0, 'finished': '2026-11-23 00:53:17'}
Leakage tests passed: True


## 3. Quick LightGBM vs the best baseline (Gate G1, part 2)
Default settings, 300,000-row training subset, early stopping on validation. No tuning yet; this only asks
"is there signal beyond the simple rules?"

In [ ]:
quick, beats = [], {}
for h in ev.HORIZONS:
    tr, va = ev.load_table(h, ["train"]), ev.load_table(h, ["val"])
    sub = mo.month_stratified_subset(tr, mo.TUNE_ROWS)
    Xs, ys, _ = mo.xy(sub); Xv, yv, scv = mo.xy(va)
    m = mo.make_model("lgbm", mo.DEFAULT_PARAMS["lgbm"], early_stop=True)
    t0 = time.time(); m, it, _ = mo.fit_model("lgbm", m, Xs, ys, Xv, yv, scv, early_stop=True); fit_s = time.time() - t0
    pred = m.predict(Xv) * scv
    s = M.score(va["y_raw"], pred)
    bb = ev.baseline_table(va).loc[best_b[h]]
    beats[h] = s["MAE"] < bb["MAE"]
    quick.append({"horizon_min": h, "LightGBM MAE": s["MAE"], "best baseline": best_b[h], "baseline MAE": bb["MAE"],
                  "skill": M.skill(s["MAE"], bb["MAE"]), "trees": it, "fit_s": fit_s})
display(pd.DataFrame(quick).round(3))

## 4. Compute projection (Gate G1, part 3)
Each model is timed on a 40,000-row sample, then extrapolated to the real protocol (tuning on 300k rows, one fit on
all training rows, one refit on train + validation, both horizons; LightGBM refit three times for seeds).
This is a **rough** estimate: real times depend on your CPU and on how many trees early stopping keeps.

In [ ]:
proj = []
for name in mo.MODELS:
    pr = mo.time_probe(name)
    proj.append({"model": mo.LABEL[name], "probe_s": pr["probe_s"], "probe_trees": pr["probe_trees"], "projected_hours": mo.project_hours(name, pr)})
pj = pd.DataFrame(proj).sort_values("projected_hours", ascending=False)
display(pj.round(2))
total_h = float(pj["projected_hours"].sum())
print(f"Projected total compute: {total_h:.1f} hours  (gate limit 12 h; Colab can take the slowest jobs)")

## 5. Gate G1 verdict

In [ ]:
checks = {"Both leakage tests pass": tests_ok,
          "LightGBM beats best baseline at 15 min": bool(beats[15]),
          "LightGBM beats best baseline at 60 min": bool(beats[60]),
          "Projected compute under 12 h": total_h < 12}
for k, v in checks.items():
    print(("PASS  " if v else "FAIL  ") + k)
g1 = all(checks.values())
(ev.RESULTS / "gate_g1.json").write_text(json.dumps({"passed": g1, "checks": checks, "projected_hours": total_h}, indent=2))
print("\nGATE G1:", "PASSED - go to notebook 03" if g1 else "NOT PASSED - fix the failing item before continuing")